In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
def make_rag_messages(question, history, chunks):
    
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [{"role": "system", "content": system_prompt}]

    for user_msg, assistant_msg in history:
        if user_msg:
            messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": question})
    return messages

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="google/embeddinggemma-300m")
DB_NAME = "chroma_langchain_db"

In [ ]:
retriever = Chroma(
    collection_name="context_collection",
    embedding_function=embeddings,
    persist_directory=DB_NAME
).as_retriever()

In [ ]:
from litellm import completion
MODEL = "gpt-4.1-nano"

In [ ]:
def chat(question, history):
    chunks = retriever.invoke(question)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content
    

In [ ]:
from _2_2__rag_question_answering import chat

question = "What new policy regarding travel and teleconferencing is GreenTech International implementing from November 15th, 2023?"

print(chat(question, []))

In [ ]:
import gradio as gr

gr.ChatInterface(chat).launch()